In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_score, recall_score, classification_report
import os
import warnings
warnings.filterwarnings('ignore')
os.chdir('/home/pgcp-ai/MachineLearning/Datasets')

In [2]:
hr = pd.read_csv("HR_comma_sep.csv")
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [3]:
hr.isna().sum()

satisfaction_level       0
last_evaluation          0
number_project           0
average_montly_hours     0
time_spend_company       0
Work_accident            0
left                     0
promotion_last_5years    0
Department               0
salary                   0
dtype: int64

In [4]:
hr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14995 entries, 0 to 14994
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   satisfaction_level     14995 non-null  float64
 1   last_evaluation        14995 non-null  float64
 2   number_project         14995 non-null  int64  
 3   average_montly_hours   14995 non-null  int64  
 4   time_spend_company     14995 non-null  int64  
 5   Work_accident          14995 non-null  int64  
 6   left                   14995 non-null  int64  
 7   promotion_last_5years  14995 non-null  int64  
 8   Department             14995 non-null  object 
 9   salary                 14995 non-null  object 
dtypes: float64(2), int64(6), object(2)
memory usage: 1.1+ MB


In [5]:
X, y = hr.drop('left', axis = 1), hr['left']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 26)

In [7]:
ohe = OneHotEncoder(sparse_output=False, drop='first').set_output(transform = 'pandas')

In [8]:
transformer = ColumnTransformer(transformers=[('OHE',ohe,make_column_selector(dtype_include=object)),
                                            ]
                               ,remainder='passthrough',
                               verbose_feature_names_out=False
                               ).set_output(transform='pandas')

In [9]:
X_train_trans = transformer.fit_transform(X_train)
X_test_trans = transformer.transform(X_test)

In [10]:
lr = LogisticRegression()
lr.fit(X_train_trans, y_train)

LogisticRegression()

In [11]:
solvers=['lbfgs','liblinear','newton-cg','newton-cholesky','sag','saga']
Cs = np.linspace(0.001,5,20)
scores = []
for s in solvers:
    for c in Cs:
        lr = LogisticRegression(solver=s,C=c)
        lr.fit(X_train_trans,y_train)
        y_pred = lr.predict(X_test_trans)
        scores.append([s,c,f1_score(y_test,y_pred,pos_label=1)])


In [12]:
df_scores = pd.DataFrame(scores,columns=['Solver','C','Score'])

In [13]:
df_scores.sort_values('Score',ascending=False)

,Solver,C,Score
11,lbfgs,2.895158,0.505983
14,lbfgs,3.684474,0.495707
17,lbfgs,4.473789,0.489068
10,lbfgs,2.632053,0.486797
6,lbfgs,1.579632,0.484779
...,...,...,...
100,saga,0.001000,0.000000
80,sag,0.001000,0.000000
40,newton-cg,0.001000,0.000000
20,liblinear,0.001000,0.000000


In [14]:
y.value_counts()

left
0    11428
1     3567
Name: count, dtype: int64

In [15]:
tst = pd.read_csv('tst_hr.csv')
tst

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,Department,salary
0,0.11,0.88,7,272,4,0,0,sales,medium
1,0.72,0.87,5,223,5,0,0,sales,low
2,0.37,0.52,2,159,3,0,0,sales,low
3,0.41,0.50,2,153,3,0,0,sales,low
4,0.67,0.61,3,202,2,0,0,technical,medium
5,0.76,0.62,3,150,2,1,0,technical,high
6,0.19,0.78,5,156,6,0,0,technical,medium
7,0.52,0.73,2,233,3,0,0,technical,medium
8,0.66,0.59,5,262,2,0,0,technical,medium
9,0.95,0.67,3,183,3,0,0,support,medium


In [23]:
X = transformer.fit_transform(X)

In [24]:
bm = LogisticRegression(solver = 'lbfgs', C = 2.895158)
bm.fit(X, y)

LogisticRegression(C=2.895158)

In [25]:
tst = transformer.transform(tst)

In [26]:
tst['Predicted_left'] = bm.predict(tst)
tst

,Department_RandD,Department_accounting,Department_hr,Department_management,Department_marketing,Department_product_mng,Department_sales,Department_support,Department_technical,salary_low,salary_medium,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,Predicted_left
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.11,0.88,7,272,4,0,0,1
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.72,0.87,5,223,5,0,0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.37,0.52,2,159,3,0,0,1
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.41,0.50,2,153,3,0,0,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.67,0.61,3,202,2,0,0,0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.76,0.62,3,150,2,1,0,0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.19,0.78,5,156,6,0,0,1
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.52,0.73,2,233,3,0,0,0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.66,0.59,5,262,2,0,0,0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.95,0.67,3,183,3,0,0,0


In [27]:
tst["Predicted_left"].value_counts()

Predicted_left
0    11
1     4
Name: count, dtype: int64